In [3]:
import pandas as pd
import pymysql
from tqdm import tqdm
import pickle

In [ ]:
def connect_db():
    return pymysql.connect(
        host="127.0.0.1",
        user='root',
        password='240812',
        database='lol_data',
        port=3307
    )

def get_champion_dict(conn):
    champion_dict_query = "SELECT champion_id, champion_name FROM champion_dict"
    return pd.read_sql_query(champion_dict_query, conn)

def save_champion_dict_to_pkl():
    conn 
# 참여자 수 계산 함수
def count_individual_participants(participant_str, assist_str):
    participants = set()
    if pd.notna(participant_str):
        participants.update(participant_str.split(','))
    if pd.notna(assist_str):
        for assist in assist_str.split(','):
            i = 0
            while i < len(assist):
                if i < len(assist) - 1 and assist[i:i+2] in map(str, range(1, 11)):
                    participants.add(assist[i:i+2])
                    i += 2
                else:
                    participants.add(assist[i])
                    i += 1
    return len(participants)

In [77]:
def get_champion_main_stats(conn):
    main_stats_query = """
    SELECT 
        cp.champion_id,
        FLOOR(cs.timestamp / 60000) AS minute,
        ROUND(SUM(cs.tdd_to_champion), 1) AS total_tdd_to_champion,
        ROUND(SUM(cs.total_damage_taken), 1) AS total_damage_taken,
        ROUND(SUM(cs.total_gold), 1) AS total_gold,
        ROUND(SUM(cs.xp), 1) AS total_xp
    FROM 
        champion_stat_per_timestamp cs
    JOIN 
        champion_participant_id cp ON cs.match_id = cp.match_id AND cs.participant_id = cp.participant_id
    GROUP BY 
        cp.champion_id, FLOOR(cs.timestamp / 60000)
    ORDER BY 
        cp.champion_id, minute;
    """
    return pd.read_sql_query(main_stats_query, conn)

def save_main_stats_to_pkl():
    conn = connect_db()
 
    champion_dict = get_champion_dict(conn)
    main_stats = get_champion_main_stats(conn)
    conn.close()
    
    champion_dataframes = {}
    for champion_id, champion_name in tqdm(champion_dict.items(), desc="Processing Champions"):
        champion_df = main_stats[main_stats['champion_id'] == champion_id].drop(columns=['champion_id'])
        champion_dataframes[champion_name] = champion_df
    with open("champion_stats.pkl", "wb") as f:
        pickle.dump(champion_dataframes, f)
    print("Champion stats data saved to champion_stats.pkl")

save_main_stats_to_pkl()

C:\Users\dgjja\AppData\Local\Temp\ipykernel_14872\754658891.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(champion_dict_query, conn)
C:\Users\dgjja\AppData\Local\Temp\ipykernel_14872\3362051903.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(main_stats_query, conn)
Processing Champions: 0it [00:00, ?it/s]


TypeError: unhashable type: 'Series'

In [ ]:

# BUILDING_KILL 데이터 가져오기
def get_building_kills(conn):
    building_kill_query = """
    SELECT 
        cp.champion_id,
        bk.match_id,
        FLOOR(bk.timestamp / 60000) AS minute,
        bk.participant_id AS participants_id,
        GROUP_CONCAT(bk.assist_id) AS assists
    FROM 
        BUILDING_KILL bk
    JOIN 
        champion_participant_id cp ON bk.match_id = cp.match_id
    GROUP BY 
        cp.champion_id, bk.match_id, FLOOR(bk.timestamp / 60000);
    """
    return pd.read_sql_query(building_kill_query, conn)

def get_champion_kills(conn):
    champion_kill_query = """
    SELECT 
        cp.champion_id,
        ck.match_id,
        FLOOR(ck.timestamp / 60000) AS minute,
        ck.participant_id AS participant_id,
        GROUP_CONCAT(ck.assist_id) AS assists
    FROM 
        CHAMPION_KILL ck
    JOIN 
        champion_participant_id cp ON ck.match_id = cp.match_id
    GROUP BY 
        cp.champion_id, ck.match_id, FLOOR(ck.timestamp / 60000);
    """
    return pd.read_sql_query(champion_kill_query, conn)


# 포탑 방패 파괴 이벤트 데이터 가져오기
def get_turret_plates(conn):
    turret_plate_query = """
    SELECT 
        cp.champion_id,
        tp.match_id,
        FLOOR(tp.timestamp / 60000) AS minute,
        COUNT(tp.participant_id) AS turret_plate_destroys
    FROM 
        TURRET_PLATE_DESTROYED tp
    JOIN 
        champion_participant_id cp ON tp.match_id = cp.match_id AND tp.participant_id = cp.participant_id
    GROUP BY 
        cp.champion_id, tp.match_id, FLOOR(tp.timestamp / 60000);
    """
    return pd.read_sql_query(turret_plate_query, conn)

def get_ward_kills(conn):
    ward_kill_query = """
    SELECT 
        cp.champion_id,
        wk.match_id,
        FLOOR(wk.timestamp / 60000) AS minute,
        COUNT(wk.participant_id) AS ward_kills
    FROM 
        WARD_KILL wk
    JOIN 
        champion_participant_id cp ON wk.match_id = cp.match_id AND wk.participant_id = cp.participant_id
    GROUP BY 
        cp.champion_id, wk.match_id, FLOOR(wk.timestamp / 60000);
    """
    return pd.read_sql_query(ward_kill_query, conn)

def get_ward_placed(conn):
    ward_placed_query = """
    SELECT 
        cp.champion_id,
        wp.match_id,
        FLOOR(wp.timestamp / 60000) AS minute,
        COUNT(wp.participant_id) AS ward_placed
    FROM 
        WARD_PLACED wp
    JOIN 
        champion_participant_id cp ON wp.match_id = cp.match_id AND wp.participant_id = cp.participant_id
    GROUP BY 
        cp.champion_id, wp.match_id, FLOOR(wp.timestamp / 60000);
    """
    return pd.read_sql_query(ward_placed_query, conn)

# ELITE_MONSTER_KILL 데이터 가져오기
def get_elite_monster_kills(conn):
    elite_monster_kill_query = """
    SELECT 
        cp.champion_id,
        emk.match_id,
        FLOOR(emk.timestamp / 60000) AS minute,
        emk.participant_id AS participant_id,
        GROUP_CONCAT(emk.assist_id) AS assists
    FROM 
        ELITE_MONSTER_KILL emk
    JOIN 
        champion_participant_id cp ON emk.match_id = cp.match_id
    GROUP BY 
        cp.champion_id, emk.match_id, FLOOR(emk.timestamp / 60000);
    """
    return pd.read_sql_query(elite_monster_kill_query, conn)
conn = connect_db()
champion_dict = get_champion_dict(conn)
main_stats = get_champion_main_stats(conn)
tower_kills = get_building_kills(conn)
champion_kills = get_champion_kills(conn)
turret_plates = get_turret_plates(conn)
ward_kills = get_ward_kills(conn)
ward_placed = get_ward_placed(conn)
elite_monsters = get_elite_monster_kills(conn)
conn.close()


C:\Users\dgjja\AppData\Local\Temp\ipykernel_14872\754658891.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(champion_dict_query, conn)
C:\Users\dgjja\AppData\Local\Temp\ipykernel_14872\2113834418.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(main_stats_query, conn)
C:\Users\dgjja\AppData\Local\Temp\ipykernel_14872\2113834418.py:38: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(building_kill_query, conn)


DatabaseError: Execution failed on sql '
    SELECT 
        cp.champion_id,
        bk.match_id,
        FLOOR(bk.timestamp / 60000) AS minute,
        bk.participant_id AS participants_id,
        GROUP_CONCAT(bk.assist_id) AS assists
    FROM 
        BUILDING_KILL bk
    JOIN 
        champion_participant_id cp ON bk.match_id = cp.match_id
    GROUP BY 
        cp.champion_id, bk.match_id, FLOOR(bk.timestamp / 60000);
    ': (1055, "Expression #4 of SELECT list is not in GROUP BY clause and contains nonaggregated column 'lol_data.bk.participant_id' which is not functionally dependent on columns in GROUP BY clause; this is incompatible with sql_mode=only_full_group_by")

In [75]:
tower_kills

,champion_id,match_id,minute,participants,assists
0,1,7291078705,10,3,None
1,1,7291078705,13,"4,0",None
2,1,7291241105,13,"4,0",None
3,1,7291241105,14,0,None
4,1,7291241105,16,"1,2,3,4,3","416,736,864,416,864"
...,...,...,...,...,...
3823645,950,7355447671,21,8,None
3823646,950,7355447671,22,0,16
3823647,950,7355447671,23,"0,9",None
3823648,950,7355447671,24,"7,6",4


In [72]:
# 전체 minute 범위 설정
minute_range = pd.DataFrame({'minute': range(0, 41)})
champion_dataframes = {}

for champion_id, champion_name in tqdm(champion_dict.values, desc="Processing Champions"):
    # 메인 스탯 데이터 필터링
    champion_main_stats = main_stats[main_stats['champion_id'] == champion_id].copy()
    champion_main_stats = champion_main_stats.drop(columns=['champion_id'])

    # 포탑 파괴 (assist_id 포함) 필터링 및 참여자 수 계산
    champion_tower_kills = tower_kills[tower_kills['champion_id'] == champion_id].copy()
    champion_tower_kills['tower_kills'] = champion_tower_kills.apply(
        lambda row: count_individual_participants(row['participant_id'], row['assist_id']),
        axis=1
    )
    
    # 챔피언 킬 (assist_id 포함) 필터링 및 참여자 수 계산
    champion_champion_kills = champion_kills[champion_kills['champion_id'] == champion_id].copy()
    champion_champion_kills['champion_kills'] = champion_champion_kills.apply(
        lambda row: count_individual_participants(row['participant_id'], row['assist_id']),
        axis=1
    )

    # 포탑 방패 파괴 (assist_id 없음) 필터링
    champion_turret_plates = turret_plates[turret_plates['champion_id'] == champion_id].copy()
    champion_turret_plates['turret_plate_destroys'] = champion_turret_plates.groupby(['match_id', 'minute']).size().groupby('minute').cumsum()

    # 와드 제거 (assist_id 없음) 필터링
    champion_ward_kills = ward_kills[ward_kills['champion_id'] == champion_id].copy()
    champion_ward_kills['ward_kills'] = champion_ward_kills.groupby(['match_id', 'minute']).size().groupby('minute').cumsum()

    # 와드 설치 (assist_id 없음) 필터링
    champion_ward_placed = ward_placed[ward_placed['champion_id'] == champion_id].copy()
    champion_ward_placed['ward_placed'] = champion_ward_placed.groupby(['match_id', 'minute']).size().groupby('minute').cumsum()

    # 엘리트 몬스터 처치 (assist_id 포함) 필터링 및 참여자 수 계산
    champion_elite_monsters = elite_monsters[elite_monsters['champion_id'] == champion_id].copy()
    champion_elite_monsters['elite_monster_kills'] = champion_elite_monsters.apply(
        lambda row: count_individual_participants(row['participant_id'], row['assist_id']),
        axis=1
    )

    # 매치별 누적된 데이터를 분 단위 평균으로 변환
    minute_avg_tower_kills = champion_tower_kills.groupby('minute')['tower_kills'].mean().reset_index().round(1)
    minute_avg_kills = champion_champion_kills.groupby('minute')['champion_kills'].mean().reset_index().round(1)
    minute_avg_plates = champion_turret_plates.groupby('minute')['turret_plate_destroys'].mean().reset_index().round(1)
    minute_avg_ward_kills = champion_ward_kills.groupby('minute')['ward_kills'].mean().reset_index().round(1)
    minute_avg_ward_placed = champion_ward_placed.groupby('minute')['ward_placed'].mean().reset_index().round(1)
    minute_avg_elite_monsters = champion_elite_monsters.groupby('minute')['elite_monster_kills'].mean().reset_index().round(1)

    # 메인 스탯과 모든 이벤트 데이터를 minute 범위와 결합, 결측값을 0으로 채움
    combined_df = minute_range.merge(champion_main_stats, on='minute', how='left')
    combined_df = combined_df.merge(minute_avg_tower_kills, on='minute', how='left').fillna(0)
    combined_df = combined_df.merge(minute_avg_kills, on='minute', how='left').fillna(0)
    combined_df = combined_df.merge(minute_avg_plates, on='minute', how='left').fillna(0)
    combined_df = combined_df.merge(minute_avg_ward_kills, on='minute', how='left').fillna(0)
    combined_df = combined_df.merge(minute_avg_ward_placed, on='minute', how='left').fillna(0)
    combined_df = combined_df.merge(minute_avg_elite_monsters, on='minute', how='left').fillna(0)

    # 최종 챔피언별 데이터 저장
    champion_dataframes[champion_name] = combined_df

# 최종 데이터 저장
with open("champion_stats.pkl", "wb") as f:
    pickle.dump(champion_dataframes, f)
print("Champion stats data saved to champion_stats.pkl")


Processing Champions:   0%|          | 0/168 [00:00<?, ?it/s]


KeyError: 'participant_id'

In [2]:
with open('data/champion_main_stats.pkl', 'rb') as file:
    champion_main_stats_data = pickle.load(file)

NameError: name 'pickle' is not defined

In [65]:
champion_stats_data['카타리나']

,minute,avg_tdd_to_champion,avg_total_damage_taken,avg_total_gold,avg_xp,tower_kills,champion_kills,turret_plate_destroys,ward_kills,ward_placed,elite_monster_kills
0,0,0.0,0.0,1091000.0,0.0,0.0,3.6,0.0,3.5,69.0,0.0
1,1,32930.0,50290.0,1100769.0,519.0,0.0,3.6,0.0,5.0,363.0,0.0
2,2,257841.0,441046.0,1277701.0,418223.0,0.0,3.2,1.0,1.0,106.9,0.0
3,3,943969.0,1504610.0,1955021.0,1792732.0,0.0,4.0,1.0,4.0,266.6,0.0
4,4,1872788.0,2680494.0,2733535.0,2923417.0,0.0,4.0,1.0,23.2,570.2,0.0
5,5,2793984.0,3906817.0,3564576.0,4072420.0,0.0,3.8,9.5,39.8,422.4,62.6
6,6,3751037.0,5123205.0,4413235.0,5244898.0,0.0,4.1,25.0,58.4,302.6,2135.8
7,7,4824362.0,6338978.0,5241892.0,6234871.0,2.6,4.0,50.9,78.9,499.8,996.9
8,8,5968468.0,7634561.0,6179596.0,7421220.0,1.4,3.9,63.0,92.7,370.8,470.9
9,9,7089643.0,8912055.0,7092542.0,8556742.0,1.5,4.0,96.4,134.1,253.0,399.7
